# 04 - Two-Way Fixed Effects DiD

State + year FE removes time-invariant state characteristics and common year effects. This is the standard DiD estimator but is biased under staggered adoption with heterogeneous effects. Notebook 05 corrects this.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"

fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
fars_reg['post'] = (
    fars_reg['retail_sales_year'].notna() &
    (fars_reg['year'] >= fars_reg['retail_sales_year'])
).astype(float)

# Exclude states with legalization outside 2010-2022 window
in_window = leg[leg['retail_sales_year'].between(2010,2022)]['state'].tolist()
never = leg[leg['retail_sales_year'].isna()]['state'].tolist()
fars_twfe = fars_reg[fars_reg['state'].isin(in_window + never)].copy()

print(f"TWFE sample: {fars_twfe['state'].nunique()} states")
print(f"  Treated: {len(in_window)}, Never-treated: {len(never)}")

## TWFE specification

In [ ]:
twfe_idx = fars_twfe.set_index(['state','year'])
fe = PanelOLS(
    twfe_idx[primary],
    twfe_idx[['post']],
    entity_effects=True,
    time_effects=True,
).fit(cov_type='clustered', cluster_entity=True)

att = fe.params['post']
ci  = fe.conf_int().loc['post']
print(f"TWFE ATT (pooled): {att:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
print(f"Interpretation: {att:.4f} change in {primary.replace('_',' ')} per 100k after legalization")
print()
print(fe.summary.tables[1])

## Why TWFE is biased here

With 15+ treatment cohorts from 2014-2022, TWFE uses **already-treated states as implicit controls** for later-treated states. If early adopters (CO, WA) had decreasing effects over time, TWFE can produce coefficients with the wrong sign even when the true effect is always negative.

Callaway-Sant'Anna (notebook 05) fixes this by restricting each cohort's comparison group to never-treated or not-yet-treated states.

In [ ]:
# Visualize: do effects look like they differ by legalization year?
from linearmodels.panel import PanelOLS

cohort_results = {}
for yr in sorted(leg['retail_sales_year'].dropna().unique()):
    yr = int(yr)
    if yr > 2022: continue
    cohort_states = leg[leg['retail_sales_year']==yr]['state'].tolist()
    sub = fars_reg[fars_reg['state'].isin(cohort_states + never)].copy()
    sub['post_cohort'] = (sub['state'].isin(cohort_states) & (sub['year'] >= yr)).astype(float)
    try:
        idx = sub.set_index(['state','year'])
        fe_c = PanelOLS(idx[primary], idx[['post_cohort']],
                        entity_effects=True, time_effects=True).fit(
                        cov_type='clustered', cluster_entity=True)
        cohort_results[yr] = fe_c.params['post_cohort']
    except Exception:
        pass

print("Cohort-specific TWFE estimates (preliminary — C-S in next notebook):")
for yr, b in sorted(cohort_results.items()):
    print(f"  {yr} cohort: {b:+.4f}")
print("\nVariation across cohorts = evidence of heterogeneous effects = TWFE bias.")